In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import (confusion_matrix,accuracy_score,
recall_score,f1_score,roc_auc_score,precision_score)
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

In [10]:
df=pd.read_csv(r'C:\Users\USER\Desktop\5th sem\project1_daibatise\data\processed\mohammed_ajlan_p1w1_features.csv')

In [11]:
feature_cols = ["age","pregnancies","glucose","blood_pressure",
                "skin_thickness","insulin","bmi","diabetes_pedigree",
                "bmi_category","glucose_band"]
x = df[feature_cols].copy()
y = df["outcome"]
x
x=pd.get_dummies(x,drop_first=True)
x.shape
x_train,x_test,y_train,y_test= train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y)


In [15]:
candidates = {
    "LogReg": ("scaled",LogisticRegression(max_iter=1000)),
    "Decision Tree": ("raw",DecisionTreeClassifier(max_depth=3, random_state=42)),
    "KNN": ("scaled",KNeighborsClassifier(n_neighbors=25)),
    "SVC balanced": ("scaled",SVC(class_weight="balanced", random_state=42)),
    "RF balanced": ("raw",RandomForestClassifier(max_depth=5,class_weight="balanced",random_state=42))
}

TARGET_RECALL, GUARD_F1 = 0.70, 0.50  

print("rows:", len(df), "| train:", len(x_train), "| test:", len(x_test))
print("diabetic in test set:", int(y_test.sum()))
print("features after get_dummies:", x.shape[1])
print("candidates:", list(candidates))
print("target: recall >=", TARGET_RECALL, "with F1 >=", GUARD_F1)

rows: 950 | train: 760 | test: 190
diabetic in test set: 55
features after get_dummies: 13
candidates: ['LogReg', 'Decision Tree', 'KNN', 'SVC balanced', 'RF balanced']
target: recall >= 0.7 with F1 >= 0.5


resplit the data and watch the score move

In [19]:
rows = []

for rs in [0, 1, 7, 42, 99]:
    Xa, Xb, ya, yb = train_test_split(
        x, y,
        test_size=0.2,
        random_state=rs,
        stratify=y
    )

    row = {"split": rs}

    for name, (kind, model) in candidates.items():
        m = clone(model)                     

        if kind == "scaled":
            sc = StandardScaler().fit(Xa)
            m.fit(sc.transform(Xa), ya)
            pred = m.predict(sc.transform(Xb))

        else:
            m.fit(Xa, ya)
            pred = m.predict(Xb)
            row[name] = round(recall_score(yb,pred),3)
            rows.append(row)

            splits = pd.DataFrame(rows).set_index("split")
print(splits)

spread = (splits.max() - splits.min()).round(3)
print("\nSpread (best - worst):\n", spread.sort_values(ascending=False))
            

    

       Decision Tree  RF balanced
split                            
0              0.091        0.600
0              0.091        0.600
1              0.382        0.618
1              0.382        0.618
7              0.182        0.582
7              0.182        0.582
42             0.291        0.655
42             0.291        0.655
99             0.236        0.673
99             0.236        0.673

Spread (best - worst):
 Decision Tree    0.291
RF balanced      0.091
dtype: float64


what is k-for cross validation?

K-Fold Cross-Validation is a technique in Machine Learning used to evaluate how well a model performs by dividing a dataset into K equal parts (folds).

set up cross validation the right way